# Attention, one step at a time


**The road**

| Part | What we add | Why |
|---|---|---|
| 1 | dot-product scores | a first way to compare tokens |
| 2 | Q, K, V | make the comparison *learnable* |
| 3 | `/ sqrt(d_k)` | keep softmax from saturating |
| 4 | causal mask | stop the model from cheating |
| 5 | softmax @ V | actually mix the tokens |
| 6 | **function:** one head | first checkpoint |
| 7 | many heads | one head is a compromise |
| 8 | **function:** many heads | second checkpoint |
| 9 | KV cache | make generation fast |
| 10 | **class:** `MultiHeadAttention` | the real file |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from IPython.display import display

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

print("torch", torch.__version__)

torch 2.13.0


In [2]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Print a 2-D tensor with token names on the rows and columns."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()

def show_colored(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Display a 2-D tensor as a color-coded table inside Jupyter."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    frame = pd.DataFrame(
        mat.detach().cpu().numpy(),
        index=row_labels,
        columns=col_labels,
    )
    minimum = float(frame.min().min())
    maximum = float(frame.max().max())
    span = max(maximum - minimum, 1e-12)

    def cell_style(value):
        # Interpolate blue -> white -> red without requiring matplotlib.
        ratio = (float(value) - minimum) / span
        if ratio <= 0.5:
            amount = ratio * 2
            start, end = (59, 130, 246), (255, 255, 255)
        else:
            amount = (ratio - 0.5) * 2
            start, end = (255, 255, 255), (239, 68, 68)
        red, green, blue = (
            round(a + (b - a) * amount) for a, b in zip(start, end)
        )
        text = "white" if ratio < 0.2 or ratio > 0.8 else "black"
        return f"background-color: rgb({red}, {green}, {blue}); color: {text}"

    if title:
        print(title)
    display(
        frame.style
        .map(cell_style)
        .format(f"{{:.{decimals}f}}")
    )


## A toy sentence

The real model uses `d_model=768` and `n_heads=12`. We cannot read a 768-wide
matrix, so this notebook uses **deliberately tiny numbers**:

```
B       = 1     one sentence in the batch
T       = 5     five tokens
d_model = 8     each token is an 8-number vector
n_heads = 2     two heads
d_k     = 4     8 / 2 -> each head gets 4 numbers
```

Every shape you see below is the real shape, just small.

In [3]:
TOKENS  = ["the", "cat", "sat", "on", "mat"]

B       = 1
T       = len(TOKENS)
d_model = 8
n_heads = 2
d_k     = d_model // n_heads

x = torch.randn(B, T, d_model)

print("tokens :", TOKENS)
print("x.shape:", tuple(x.shape), " (batch, time, width)")
print()
print("x[0] -- one row per token:")
print(x)

tokens : ['the', 'cat', 'sat', 'on', 'mat']
x.shape: (1, 5, 8)  (batch, time, width)

x[0] -- one row per token:
tensor([[[-1.13, -1.15, -0.25, -0.43,  0.85,  0.69, -0.32, -2.12],
         [ 0.32, -1.26,  0.35,  0.31,  0.12,  1.24,  1.12, -0.25],
         [-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85],
         [-0.22, -0.74,  0.56,  0.26, -0.17, -0.68,  0.94,  0.49],
         [ 1.20,  0.08, -1.20, -0.00, -0.52, -0.31, -1.58,  1.71]]])


In [4]:
col_labels = [ f"d{i}" for i in range(5) ]
show(x[0][:, :5], "x, first 5 columns", col_labels=col_labels)

x, first 5 columns
             d0      d1      d2      d3      d4
   the    -1.13   -1.15   -0.25   -0.43    0.85
   cat     0.32   -1.26    0.35    0.31    0.12
   sat    -1.35   -1.70    0.57    0.79    0.60
    on    -0.22   -0.74    0.56    0.26   -0.17
   mat     1.20    0.08   -1.20   -0.00   -0.52



## compare tokens with a dot product

Right now every token vector knows **only its own word**. `x[2]` is "sat" and has
no idea that a cat is involved.

We want each token to build a *new* vector by mixing in the tokens it cares about:

```
new_sat = 0.6 * cat  +  0.3 * sat  +  0.1 * the
```

So the question is: **where do the numbers 0.6 / 0.3 / 0.1 come from?**

First attempt: two vectors that point in a similar direction have a large dot
product. So let us just compare every token with every other token.

In [5]:
naive_scores = x[0] @ x[0].T

print("x[0].shape:", tuple(x[0].shape))
print("naive_scores.shape:", tuple(naive_scores.shape))

show(naive_scores, "naive_scores[i][j] = how much token i matches token j")

x[0].shape: (5, 8)
naive_scores.shape: (5, 5)
naive_scores[i][j] = how much token i matches token j
            the     cat     sat      on     mat
   the     8.62    2.00   -1.39   -1.10   -4.91
   cat     2.00    4.77   -0.54    1.21   -2.77
   sat    -1.39   -0.54   11.98    3.61    1.41
    on    -1.10    1.21    3.61    2.59   -1.35
   mat    -4.91   -2.77    1.41   -1.35    8.67



---
## Q, K, V: give each token three roles

Instead of comparing `x` with `x`, we let the model learn **three different
projections** of every token.

| Name | The token is asking / offering | Library analogy |
|---|---|---|
| **Q**uery | "what am I looking for?" | what you want to read |
| **K**ey   | "what do I advertise?"   | the title on the book's spine |
| **V**alue | "what do I hand over if picked?" | the content inside the book |

You search with your **query**, you match against **keys**, and you walk away
with **values**.

Each is one `nn.Linear`. In the real file:

```python
self.W_q = nn.Linear(config.d_model, config.d_model, bias=config.qkv_bias)
```

`qkv_bias` is `False` in `config.py`: a bias shifts Q/K/V but changes nothing
about what attention computes, so it is left out.

In [6]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model)

q = W_q(x)
k = W_k(x)
v = W_v(x)

mask = torch.tril(
    torch.ones(T, T, dtype=torch.bool)
)

print("x.shape:", tuple(x.shape))
print("q.shape:", tuple(q.shape))
print("k.shape:", tuple(k.shape))
print("v.shape:", tuple(v.shape))
print()

x.shape: (1, 5, 8)
q.shape: (1, 5, 8)
k.shape: (1, 5, 8)
v.shape: (1, 5, 8)



In [7]:
print(x[0][2])
print(q[0][2])
print(k[0][2])

tensor([-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85])
tensor([ 0.24, -0.66, -0.16, -0.36,  0.03, -0.97,  0.44, -0.29], grad_fn=<SelectBackward0>)
tensor([-0.35,  0.41, -0.05,  0.72, -0.07,  0.05,  1.05,  0.49], grad_fn=<SelectBackward0>)


Now compare **queries against keys** instead of `x` against `x`.

`k.transpose(-2, -1)` swaps the last two axes: `(B, T, d_model) -> (B, d_model, T)`.


We use `-2, -1` rather than `1, 2` so the same line keeps working later, when a
head axis appears in front.

In [8]:
scores = q @ k.transpose(-2, -1)

print("scores.shape:", tuple(scores.shape))
print()
show(scores[0], "scores = q @ k^T")

scores.shape: (1, 5, 5)

scores = q @ k^T
            the     cat     sat      on     mat
   the     0.70   -1.33    0.45   -0.22    0.33
   cat    -0.69   -0.17   -1.03   -0.20    0.78
   sat    -0.51   -0.31   -0.34   -0.35    0.07
    on    -0.45    0.47   -0.55   -0.03   -0.27
   mat    -0.87   -0.58   -0.46   -0.53    1.74



---
## Scale by `sqrt(d_k)`

We are about to push these scores through a softmax. A dot product over many
dimensions produces large numbers, and softmax of large numbers collapses into a
single winner.

Let us see that happen.

In [9]:
def row(t, style="{:6.2f}"):
    return "[" + "  ".join(style.format(v) for v in t.tolist()) + "]"

small = torch.tensor([0.5, 1.2, 0.8])
big = small * 16

print("scores", row(small), "-> softmax", row(F.softmax(small, dim=-1), "{:6.1%}"))
print("scores", row(big),   "-> softmax", row(F.softmax(big,   dim=-1), "{:6.1%}"))

scores [  0.50    1.20    0.80] -> softmax [ 22.9%   46.1%   30.9%]
scores [  8.00   19.20   12.80] -> softmax [  0.0%   99.8%    0.2%]


In [10]:
scaled = scores / d_k ** 0.5

print("std of raw scores   :", round(scores.std().item(), 3))
print("std of scaled scores:", round(scaled.std().item(), 3))
print()
show(scaled[0], "scaled = scores / sqrt(d_k)")

std of raw scores   : 0.648
std of scaled scores: 0.324

scaled = scores / sqrt(d_k)
            the     cat     sat      on     mat
   the     0.35   -0.67    0.22   -0.11    0.16
   cat    -0.34   -0.09   -0.51   -0.10    0.39
   sat    -0.26   -0.15   -0.17   -0.18    0.04
    on    -0.23    0.24   -0.27   -0.01   -0.14
   mat    -0.44   -0.29   -0.23   -0.26    0.87



---
## The causal mask: no peeking at the future

Our model's job is to predict the **next** token. If, while processing "sat", the
model could look at "mat", it would be reading the answer off the page. At
generation time the future does not exist, so a model trained that way would
collapse.

So we forbid every token from seeing anything to its right.

`torch.tril` ("**tri**angle, **l**ower") builds exactly that pattern.

In [11]:
# example

max_seq_len = 16

sent = torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)
sent_mask = torch.tril(sent)

print("sent.shape:", tuple(sent.shape))
print("mask.shape:", tuple(sent_mask.shape))

show(sent[:T, :T].int())
print()
show(sent_mask[:T, :T].int())

sent.shape: (16, 16)
mask.shape: (16, 16)
            the     cat     sat      on     mat
   the     1.00    1.00    1.00    1.00    1.00
   cat     1.00    1.00    1.00    1.00    1.00
   sat     1.00    1.00    1.00    1.00    1.00
    on     1.00    1.00    1.00    1.00    1.00
   mat     1.00    1.00    1.00    1.00    1.00


            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00



In [12]:
# Let's apply



masked = scaled.masked_fill(~mask, float("-inf"))

show(masked[0], "scaled scores after masked_fill")

scaled scores after masked_fill
            the     cat     sat      on     mat
   the     0.35    -inf    -inf    -inf    -inf
   cat    -0.34   -0.09    -inf    -inf    -inf
   sat    -0.26   -0.15   -0.17    -inf    -inf
    on    -0.23    0.24   -0.27   -0.01    -inf
   mat    -0.44   -0.29   -0.23   -0.26    0.87



---
# Softmax, then mix the values

`dim=-1` is important: we normalise **across the keys**, i.e. along each row.
"Given this query, how do I spread 100% of my attention over the tokens I am
allowed to see?"

In [13]:
weights = F.softmax(masked, dim=-1) # (B, T, T)
out = weights @ v # (B, T, T) @ (B, T, d_model) => (B, T, d_model)

print("weights.shape:", tuple(weights.shape))
print("v.shape      :", tuple(v.shape))
print("out.shape    :", tuple(out.shape), " <- same shape we started with")
print()
show(weights[0], "attention weights (each row sums to 1)")

weights.shape: (1, 5, 5)
v.shape      : (1, 5, 8)
out.shape    : (1, 5, 8)  <- same shape we started with

attention weights (each row sums to 1)
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.44    0.56    0.00    0.00    0.00
   sat     0.31    0.35    0.34    0.00    0.00
    on     0.21    0.33    0.20    0.26    0.00
   mat     0.12    0.14    0.15    0.14    0.45



---
# Checkpoint: wrap one head in a function

Nothing new below. We are only collecting Parts 2-5 into one place.

In [14]:
def single_head_attention(x, W_q, W_k, W_v, mask):
    """One attention head.

    x -> (B, T, d_model);  returns (B, T, d_model) and the attention weights.
    """

    B, T, d_model = x.shape

    q = W_q(x)                                          # Part 2
    k = W_k(x)
    v = W_v(x)

    scores = q @ k.transpose(-2, -1)                    # Part 2
    scores = scores / d_model**0.5                      # Part 3

    causal = mask[:T, :T]                               # Part 4
    scores = scores.masked_fill(~causal, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = weights @ v

    return out, weights

In [15]:
head_out, head_w = single_head_attention(x, W_q, W_k, W_v, mask)

print("in :", tuple(x.shape))
print("out:", tuple(head_out.shape))

in : (1, 5, 8)
out: (1, 5, 8)


---
# One head is not enough

A sentence contains many kinds of relation at the same time:

* subject -> verb        ("cat" -> "sat")
* adjective -> noun
* pronoun -> what it refers to

One head produces **one** `(T, T)` weight matrix, so it has to compromise between
all of them.

Are two heads really different, or do they learn the same thing? Let us test it:
build two heads with independent random weights and compare their patterns.

In [16]:
head_a, head_b = None, None

torch.manual_seed(101)
head_a = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

torch.manual_seed(202)
head_b = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

o_a, w_a = single_head_attention(x, *head_a, mask)
o_b, w_b = single_head_attention(x, *head_b, mask)

show(w_a[0], "head A")
show(w_b[0], "head B")

head A
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.48    0.52    0.00    0.00    0.00
   sat     0.44    0.38    0.18    0.00    0.00
    on     0.28    0.25    0.23    0.24    0.00
   mat     0.35    0.25    0.07    0.15    0.17

head B
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.46    0.54    0.00    0.00    0.00
   sat     0.55    0.31    0.14    0.00    0.00
    on     0.26    0.27    0.22    0.25    0.00
   mat     0.25    0.12    0.08    0.12    0.42



## How to run many heads cheaply

**The obvious way:** make 12 small `nn.Linear(768, 64)` layers and loop over them.
Correct, but that is 12 separate small matmuls -- slow on a GPU.

**The trick used in the file:** keep *one* full-width `nn.Linear(768, 768)` per
role, and **carve** the heads out of its output by reshaping.

```
(B, T, 8)                         after W_q
   | .view(B, T, 2, 4)            split the width into 2 chunks of 4
(B, T, 2, 4)
   | .transpose(1, 2)             move the head axis in front of time
(B, 2, T, 4)                      now: 2 independent (T, 4) problems
```

In [17]:
# projected = W_q(x)
# print("1. after W_q            :", tuple(projected.shape))

# split = projected.view(B, T, n_heads, d_k)
# print("2. after .view          :", tuple(split.shape), " (B, T, n_heads, d_k)")

# q_heads = split.transpose(1, 2)
# print("3. after .transpose(1,2):", tuple(q_heads.shape), " (B, n_heads, T, d_k)")

q_heads = W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
print("q_heads:", tuple(q_heads.shape), " (B, n_heads, T, d_k)")

q_heads: (1, 2, 5, 4)  (B, n_heads, T, d_k)


## Checkpoint: wrap multi-head attention in a function

In [18]:
def multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads):
    """Multi-head causal self-attention.  (B, T, d_model) -> (B, T, d_model)"""

    B, T, d_model = x.shape
    d_k = d_model // n_heads

    q = W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
    k = W_k(x).view(B, T, n_heads, d_k).transpose(1, 2) # => (B, n_heads, T, d_k)
    v = W_v(x).view(B, T, n_heads, d_k).transpose(1, 2)

    scores = q @ k.transpose(-2, -1)
    scores = scores / d_k**0.5
    scores = scores.masked_fill(~mask[:T, :T], float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = weights @ v

    out = out.transpose(1, 2).contiguous().view(B, T, d_model)
    # .transpose(1, 2) ===> (B, T, n_heads, d_k)
    # .view(B, T, d_model) ===> (B, T, d_model)

    return W_o(out), weights


In [19]:

mha_out, mha_w =  multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads)


print("in :", tuple(x.shape))
print("out:", tuple(mha_out.shape))
print("weights:", tuple(mha_w.shape), " (B, n_heads, T, T)")
print()

# mha_w has shape (batch, heads, query tokens, key tokens).
# Display the 5 x 5 attention matrix of every head with a color scale.
for head_index in range(n_heads):
    show_colored(
        mha_w[0, head_index],
        title=f"Attention weights - head {head_index + 1}",
        row_labels=TOKENS,
        col_labels=TOKENS,
    )

in : (1, 5, 8)
out: (1, 5, 8)
weights: (1, 2, 5, 5)  (B, n_heads, T, T)

Attention weights - head 1


,the,cat,sat,on,mat
the,1.00,0.00,0.00,0.00,0.00
cat,0.41,0.59,0.00,0.00,0.00
sat,0.32,0.35,0.32,0.00,0.00
on,0.25,0.27,0.24,0.24,0.00
mat,0.12,0.19,0.14,0.19,0.37


Attention weights - head 2


,the,cat,sat,on,mat
the,1.00,0.00,0.00,0.00,0.00
cat,0.52,0.48,0.00,0.00,0.00
sat,0.32,0.33,0.35,0.00,0.00
on,0.21,0.31,0.21,0.27,0.00
mat,0.21,0.16,0.22,0.16,0.25


## KV Cache

In [20]:
# Feed growing prefixes, exactly as a generation loop would, and watch what
# happens to the output row for token 0 ("the").

TOKENS  = ["the", "cat", "sat", "on", "mat"]

def row(t, style="{:6.2f}"):
    return "[" + "  ".join(style.format(v) for v in t.tolist()) + "]"

for step in range(1, T + 1):
    mha_out, _ =  multi_head_attention(x[:, :step, :], W_q, W_k, W_v, W_o, mask, n_heads)
    prefix = " ".join( TOKENS[:step] )

    print(f"step: {step}")
    print(f"Prefix Input: {prefix}")
    print("Out: ", row(mha_out[0, 0, :]))
    print()
    


step: 1
Prefix Input: the
Out:  [  0.60    0.12    0.25    1.20   -0.20    0.32   -0.32    0.06]

step: 2
Prefix Input: the cat
Out:  [  0.60    0.12    0.25    1.20   -0.20    0.32   -0.32    0.06]

step: 3
Prefix Input: the cat sat
Out:  [  0.60    0.12    0.25    1.20   -0.20    0.32   -0.32    0.06]

step: 4
Prefix Input: the cat sat on
Out:  [  0.60    0.12    0.25    1.20   -0.20    0.32   -0.32    0.06]

step: 5
Prefix Input: the cat sat on mat
Out:  [  0.60    0.12    0.25    1.20   -0.20    0.32   -0.32    0.06]



In [21]:
TOKENS  = ["the", "cat", "sat", "on", "mat"]

past_length = 3
new_length = 1

# ====

past = x[:, :3, :] # "the", "cat", "sat"
new = x[:, 3:4, :] # "on"

past_k = W_k(past).view(B, past_length, n_heads, d_k).transpose(1, 2)
past_v = W_v(past).view(B, past_length, n_heads, d_k).transpose(1, 2)

kv_cache = (past_k, past_v)

# ====

q = W_q(new).view(B, new_length, n_heads, d_k).transpose(1, 2)
k = W_k(new).view(B, new_length, n_heads, d_k).transpose(1, 2)
v = W_v(new).view(B, new_length, n_heads, d_k).transpose(1, 2)

past_k, past_v = kv_cache
past_length = past_k.size(2)
seq_length = new.shape[1]

k = torch.cat([past_k, k], dim=2)
v = torch.cat([past_v, v], dim=2)

total_length = past_length + seq_length


print("past_len :", past_length,  " tokens already in the cache")
print("seq_len  :", seq_length,   " tokens fed in this step")
print("total_len:", total_length, " tokens attention can see")
print()
print("q:", tuple(q.shape), " <- 1 query   (q for 'the','cat','sat' is NOT rebuilt)")
print("k:", tuple(k.shape), " <- 4 keys    (3 reused from cache + 1 new)")
print("v:", tuple(v.shape), " <- 4 values  (3 reused from cache + 1 new)")



past_len : 3  tokens already in the cache
seq_len  : 1  tokens fed in this step
total_len: 4  tokens attention can see

q: (1, 2, 1, 4)  <- 1 query   (q for 'the','cat','sat' is NOT rebuilt)
k: (1, 2, 4, 4)  <- 4 keys    (3 reused from cache + 1 new)
v: (1, 2, 4, 4)  <- 4 values  (3 reused from cache + 1 new)


In [22]:
new.shape

torch.Size([1, 1, 8])

---

## The MyMultiHeadAttention

In [23]:
class MyMultiHeadAttention(nn.Module):
    """Every token looks at every earlier token, through several lenses at once.

    Input:  (B, T, d_model)
    Output: (B, T, d_model)
    """

    def __init__(self, config):
        super().__init__()

        self.n_heads = config.n_heads
        self.d_k = config.d_k

        # One full-width projection per role, carved into heads in forward().
        self.W_q = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_k = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_v = nn.Linear(config.d_model, config.d_model, bias=False)

        self.W_o = nn.Linear(config.d_model, config.d_model)

        mask = torch.tril(
            torch.ones(config.max_seq_len, config.max_seq_len, dtype=torch.bool)
        )

        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores / self.d_k**0.5

        scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        out = weights @ v

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(out), weights
        
        
